# Imports and installation

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    f1_score, accuracy_score, recall_score, precision_score,
    mean_absolute_error
)
import matplotlib.pyplot as plt
import seaborn as sns

print("TensorFlow version:", tf.__version__)

In [ ]:
!pip install iterative-stratification -q

from iterstrat.ml_stratifiers import (MultilabelStratifiedShuffleSplit,MultilabelStratifiedKFold)

# Preprocessing

**Cropping ROI**

In [ ]:
def crop_to_content(img, tolerance=15):
    gray   = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask   = gray > tolerance
    coords = np.argwhere(mask)
    if coords.size == 0:
        return img
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    return img[y0:y1, x0:x1]

**Loading Image**

In [ ]:
labels_path = '/kaggle/input/datasets/nuzhataisha/merged-dataset/merged_dataset/master_labels.xlsx'
base_path   = '/kaggle/input/datasets/nuzhataisha/merged-dataset/merged_dataset/master_dataset'
labels_df   = pd.read_excel(labels_path)
print("Loaded labels:", labels_df.shape)

In [ ]:
x_aceto_data, x_iodine_data, x_vascular_data = [], [], []
y_data_aceto, y_data_iod, y_data_ves, y_data_mar, y_data_les = [], [], [], [], []
extensions = ['.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.bmp']

for index, row in labels_df.iterrows():
    diag_value1 = pd.to_numeric(row['Aceto uptake'],  errors='coerce')
    diag_value2 = pd.to_numeric(row['Iodine uptake'], errors='coerce')
    diag_value3 = pd.to_numeric(row['Vessels'],       errors='coerce')
    diag_value4 = pd.to_numeric(row['Margin'],        errors='coerce')
    diag_value5 = pd.to_numeric(row['Lesion Size'],  errors='coerce')

    if any(pd.isna(v) for v in [diag_value1, diag_value2,
                                  diag_value3, diag_value4, diag_value5]):
        continue

    case_folder = str(row['Folder']).strip()

    for ext in extensions:
        aceto_path    = os.path.join(base_path, case_folder, f'001{ext}')
        iodine_path   = os.path.join(base_path, case_folder, f'002{ext}')
        vascular_path = os.path.join(base_path, case_folder, f'003{ext}')

        if os.path.exists(aceto_path) and os.path.exists(iodine_path) \
                                       and os.path.exists(vascular_path):
            img_a = cv2.imread(aceto_path)
            img_i = cv2.imread(iodine_path)
            img_v = cv2.imread(vascular_path)

            if img_a is None or img_i is None or img_v is None:
                continue

            img_a = cv2.cvtColor(img_a, cv2.COLOR_BGR2RGB)
            img_i = cv2.cvtColor(img_i, cv2.COLOR_BGR2RGB)
            img_v = cv2.cvtColor(img_v, cv2.COLOR_BGR2RGB)

            img_a = cv2.resize(crop_to_content(img_a), (224, 224))
            img_i = cv2.resize(crop_to_content(img_i), (224, 224))
            img_v = cv2.resize(crop_to_content(img_v), (224, 224))

            img_a = preprocess_input(img_a.astype("float32"))
            img_i = preprocess_input(img_i.astype("float32"))
            img_v = preprocess_input(img_v.astype("float32"))

            x_aceto_data.append(img_a)
            x_iodine_data.append(img_i)
            x_vascular_data.append(img_v)

            y_data_aceto.append(int(diag_value1))
            y_data_iod.append(int(diag_value2))
            y_data_ves.append(int(diag_value3))
            y_data_mar.append(int(diag_value4))
            y_data_les.append(int(diag_value5))
            break

X_aceto    = np.array(x_aceto_data,    dtype=np.float32)
X_iodine   = np.array(x_iodine_data,   dtype=np.float32)
X_vascular = np.array(x_vascular_data, dtype=np.float32)


y_aceto_int = np.array(y_data_aceto, dtype=np.int32)
y_iod_int   = np.array(y_data_iod,   dtype=np.int32)
y_ves_int   = np.array(y_data_ves,   dtype=np.int32)
y_mar_int   = np.array(y_data_mar,   dtype=np.int32)
y_les_int   = np.array(y_data_les,   dtype=np.int32)

y_total_int = (y_aceto_int + y_iod_int + y_ves_int + y_mar_int  + y_les_int)

y_aceto = to_categorical(y_aceto_int, num_classes=3)
y_iod   = to_categorical(y_iod_int,   num_classes=3)
y_ves   = to_categorical(y_ves_int,   num_classes=3)
y_mar   = to_categorical(y_mar_int,   num_classes=3)
y_les   = to_categorical(y_les_int,   num_classes=3)

print(f"Total samples: {len(X_aceto)}")
for name, arr in [("aceto", y_aceto_int), ("iodine", y_iod_int),
                  ("vessel", y_ves_int),  ("margin", y_mar_int),
                  ("lesion", y_les_int)]:
    counts = pd.Series(arr).value_counts().sort_index()
    print(f"{name}: {dict(counts)}")

del x_aceto_data, x_iodine_data, x_vascular_data
del y_data_aceto, y_data_iod, y_data_ves, y_data_mar, y_data_les

**Train/Test/Val Split**

In [ ]:
Y_all = np.stack([
    y_aceto_int, y_iod_int, y_ves_int,
    y_mar_int,   y_les_int
], axis=1)


msss = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.2, random_state=42
)
tv_idx, test_idx = next(msss.split(X_aceto, Y_all))

X_aceto_tv,    X_aceto_test    = X_aceto[tv_idx],    X_aceto[test_idx]
X_iodine_tv,   X_iodine_test   = X_iodine[tv_idx],   X_iodine[test_idx]
X_vascular_tv, X_vascular_test = X_vascular[tv_idx], X_vascular[test_idx]

y_aceto_tv,    y_aceto_test    = y_aceto[tv_idx],    y_aceto[test_idx]
y_iod_tv,      y_iod_test      = y_iod[tv_idx],      y_iod[test_idx]
y_ves_tv,      y_ves_test      = y_ves[tv_idx],      y_ves[test_idx]
y_mar_tv,      y_mar_test      = y_mar[tv_idx],      y_mar[test_idx]
y_les_tv,      y_les_test      = y_les[tv_idx],      y_les[test_idx]

y_aceto_int_tv, y_aceto_int_test = y_aceto_int[tv_idx], y_aceto_int[test_idx]
y_iod_int_tv,   y_iod_int_test   = y_iod_int[tv_idx],   y_iod_int[test_idx]
y_ves_int_tv,   y_ves_int_test   = y_ves_int[tv_idx],   y_ves_int[test_idx]
y_mar_int_tv,   y_mar_int_test   = y_mar_int[tv_idx],   y_mar_int[test_idx]
y_les_int_tv,   y_les_int_test   = y_les_int[tv_idx],   y_les_int[test_idx]
y_total_int_tv, y_total_int_test = y_total_int[tv_idx], y_total_int[test_idx]

Y_all_tv = Y_all[tv_idx]

print(f"Train+Val: {X_aceto_tv.shape[0]}  Test: {X_aceto_test.shape[0]}")

mskf = MultilabelStratifiedKFold(
    n_splits=3, shuffle=True, random_state=42
)
all_folds_data = []

for fold_idx, (train_idx, val_idx) in enumerate(
        mskf.split(X_aceto_tv, Y_all_tv)):
    fold_dict = {
        'fold': fold_idx,
        'train': (
            X_aceto_tv[train_idx],    X_iodine_tv[train_idx],
            X_vascular_tv[train_idx],
            y_aceto_tv[train_idx],    y_iod_tv[train_idx],
            y_ves_tv[train_idx],      y_mar_tv[train_idx],
            y_les_tv[train_idx],
            y_aceto_int_tv[train_idx], y_iod_int_tv[train_idx],
            y_ves_int_tv[train_idx],   y_mar_int_tv[train_idx],
            y_les_int_tv[train_idx],
            y_total_int_tv[train_idx],
        ),
        'val': (
            X_aceto_tv[val_idx],    X_iodine_tv[val_idx],
            X_vascular_tv[val_idx],
            y_aceto_tv[val_idx],    y_iod_tv[val_idx],
            y_ves_tv[val_idx],      y_mar_tv[val_idx],
            y_les_tv[val_idx],
            y_total_int_tv[val_idx],
        ),
        'test': (
            X_aceto_test, X_iodine_test, X_vascular_test,
            y_aceto_test, y_iod_test,   y_ves_test,
            y_mar_test,   y_les_test,
            y_total_int_test,
        )
    }
    all_folds_data.append(fold_dict)
    print(f"Fold {fold_idx}: Train={len(train_idx)}, Val={len(val_idx)}")

**Runtime augmentation**

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.20),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.05),
], name="train_augmentation")


def make_dataset(X_a, X_i, X_v,
                 y_aceto, y_iod, y_ves, y_mar, y_les,
                 y_total,                        
                 batch_size=8, training=False):
    ds = tf.data.Dataset.from_tensor_slices((
        {
            "input_aceto":    X_a.astype("float32"),
            "input_iodine":   X_i.astype("float32"),
            "input_vascular": X_v.astype("float32"),
        },
        {
            "out_aceto":  y_aceto.astype("float32"),
            "out_iodine": y_iod.astype("float32"),
            "out_vessel": y_ves.astype("float32"),
            "out_margin": y_mar.astype("float32"),
            "out_lesion": y_les.astype("float32"), 
            "y_total": y_total.astype("float32")
        }
    ))

    if training:
        ds = ds.shuffle(buffer_size=len(X_a), reshuffle_each_iteration=True)

        def augment(inputs, labels):
            combined = tf.concat([
                inputs["input_aceto"],
                inputs["input_iodine"],
                inputs["input_vascular"],
            ], axis=-1)
            combined = train_augmentation(
                tf.expand_dims(combined, 0), training=True
            )
            combined = tf.squeeze(combined, 0)
            return {
                "input_aceto":    combined[..., :3],
                "input_iodine":   combined[..., 3:6],
                "input_vascular": combined[..., 6:],
            }, labels

        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)

    return ds.batch(batch_size, drop_remainder=training).prefetch(AUTOTUNE)

# Model

**Dual stream framework**

In [ ]:
class ChannelAttention(layers.Layer):
    def __init__(self, channels, reduction=16, **kwargs):
        super().__init__(**kwargs)
        hidden = max(channels // reduction, 8)
        self.avg_pool = layers.GlobalAveragePooling2D(keepdims=True)
        self.max_pool = layers.GlobalMaxPooling2D(keepdims=True)
        self.mlp = keras.Sequential([
            layers.Conv2D(hidden, 1, use_bias=False),
            layers.ReLU(),
            layers.Conv2D(channels, 1, use_bias=False),
        ])
        self.sigmoid = layers.Activation("sigmoid")

    def call(self, x):
        return x * self.sigmoid(
            self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x))
        )


class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.conv    = layers.Conv2D(1, kernel_size, padding="same",
                                     use_bias=False)
        self.sigmoid = layers.Activation("sigmoid")

    def call(self, x):
        avg_out = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_out = tf.reduce_max(x,  axis=-1, keepdims=True)
        return x * self.sigmoid(
            self.conv(tf.concat([avg_out, max_out], axis=-1))
        )


class CBAM(layers.Layer):
    def __init__(self, channels, reduction=16, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.ca = ChannelAttention(channels, reduction=reduction)
        self.sa = SpatialAttention(kernel_size=kernel_size)

    def call(self, x):
        return self.sa(self.ca(x))


class CrossAttention2D(layers.Layer):
    def __init__(self, in_channels, attn_dim=256, **kwargs):
        super().__init__(**kwargs)
        self.attn_dim = attn_dim
        self.q_proj   = layers.Conv2D(attn_dim,    1, use_bias=False)
        self.k_proj   = layers.Conv2D(attn_dim,    1, use_bias=False)
        self.v_proj   = layers.Conv2D(attn_dim,    1, use_bias=False)
        self.out_proj = layers.Conv2D(in_channels, 1, use_bias=False)

    def call(self, query_src, key_value_src):
        q = self.q_proj(query_src)
        k = self.k_proj(key_value_src)
        v = self.v_proj(key_value_src)
        shape   = tf.shape(q)
        b, h, w = shape[0], shape[1], shape[2]
        hw      = h * w
        q = tf.reshape(q, [b, hw, self.attn_dim])
        k = tf.reshape(k, [b, hw, self.attn_dim])
        v = tf.reshape(v, [b, hw, self.attn_dim])
        scale = tf.cast(self.attn_dim, tf.float32) ** -0.5
        attn  = tf.nn.softmax(
            tf.matmul(q, k, transpose_b=True) * scale, axis=-1
        )
        out = tf.reshape(tf.matmul(attn, v), [b, h, w, self.attn_dim])
        return self.out_proj(out)


class DualCrossAttentionFusion(layers.Layer):
    def __init__(self, channels, attn_dim=256, **kwargs):
        super().__init__(**kwargs)
        self.eff_to_res = CrossAttention2D(channels, attn_dim)
        self.res_to_eff = CrossAttention2D(channels, attn_dim)
        self.bn  = layers.BatchNormalization()
        self.act = layers.ReLU()

    def call(self, eff_feat, res_feat, training=None):
        fused = (self.eff_to_res(eff_feat, res_feat) +
                 self.res_to_eff(res_feat, eff_feat))
        return self.act(self.bn(fused, training=training))


def conv_bn_relu(x, filters, kernel_size=1, stride=1, name=None):
    x = layers.Conv2D(
        filters, kernel_size, strides=stride, padding="same",
        use_bias=False, name=name + "_conv" if name else None
    )(x)
    x = layers.BatchNormalization(name=name + "_bn"  if name else None)(x)
    x = layers.ReLU(name=name + "_relu" if name else None)(x)
    return x

In [ ]:
def build_swede_model(input_shape=(224, 224, 3),
                      dropout_rate=0.4,
                      attention_dim=256,
                     n_unfreeze=20):

    input_aceto    = keras.Input(shape=input_shape, name="input_aceto")
    input_iodine   = keras.Input(shape=input_shape, name="input_iodine")
    input_vascular = keras.Input(shape=input_shape, name="input_vascular")

    C = 256

    def _freeze(base):
        for layer in base.layers[:-n_unfreeze]:
            layer.trainable = False
        for layer in base.layers[-n_unfreeze:]:
            layer.trainable = not isinstance(layer, layers.BatchNormalization)

    def make_b0(inp, suffix):
        base = keras.applications.EfficientNetB0(
            include_top=False, weights="imagenet", input_tensor=inp)
        _freeze(base)
        f28 = base.get_layer("block4a_expand_activation").output
        f14 = base.get_layer("block6a_expand_activation").output
        return keras.Model(inp, [f28, f14], name=f"B0_{suffix}")(inp)

    def make_b4(inp, suffix):
        base = keras.applications.EfficientNetB4(
            include_top=False, weights="imagenet", input_tensor=inp)
        _freeze(base)
        f28 = base.get_layer("block4a_expand_activation").output
        f14 = base.get_layer("block6a_expand_activation").output
        return keras.Model(inp, [f28, f14], name=f"B4_{suffix}")(inp)

    def fuse(p28_A, p14_A, p28_B, p14_B, name):
        f28    = DualCrossAttentionFusion(C, attention_dim,
                                          name=f"{name}_f28")(p28_A, p28_B)
        f14    = DualCrossAttentionFusion(C, attention_dim,
                                          name=f"{name}_f14")(p14_A, p14_B)
        down   = conv_bn_relu(f28, C, kernel_size=3, stride=2,
                              name=f"{name}_down")
        merged = layers.Concatenate(name=f"{name}_cat")([down, f14])
        merged = conv_bn_relu(merged, C, kernel_size=3, name=f"{name}_refine")
        merged = CBAM(C, name=f"{name}_cbam")(merged)
        pooled = layers.GlobalAveragePooling2D(name=f"{name}_gap")(merged)
        return layers.Dropout(dropout_rate, name=f"{name}_drop")(pooled)


    def head(feat, name):
        x = layers.Dense(128, activation="relu", name=f"{name}_fc")(feat)
        return layers.Dense(3, activation="softmax",
                            name=f"out_{name}")(x)


    a_xa_28, a_xa_14 = make_b4(input_aceto,  "aceto_xa")
    a_xi_28, a_xi_14 = make_b0(input_iodine, "aceto_xi")
    a_xa_28p = conv_bn_relu(a_xa_28, C, name="a_xa_28p")
    a_xa_14p = conv_bn_relu(a_xa_14, C, name="a_xa_14p")
    a_xi_28p = conv_bn_relu(a_xi_28, C, name="a_xi_28p")
    a_xi_14p = conv_bn_relu(a_xi_14, C, name="a_xi_14p")
    feat_aceto = fuse(a_xa_28p, a_xa_14p, a_xi_28p, a_xi_14p,
                      name="fuse_aceto")

   
    i_b0_28, i_b0_14 = make_b0(input_iodine, "iodine_xi_b0")
    i_b4_28, i_b4_14 = make_b4(input_iodine, "iodine_xi_b4")
    i_b0_28p = conv_bn_relu(i_b0_28, C, name="i_b0_28p")
    i_b0_14p = conv_bn_relu(i_b0_14, C, name="i_b0_14p")
    i_b4_28p = conv_bn_relu(i_b4_28, C, name="i_b4_28p")
    i_b4_14p = conv_bn_relu(i_b4_14, C, name="i_b4_14p")
    feat_iodine = fuse(i_b0_28p, i_b0_14p, i_b4_28p, i_b4_14p,
                       name="fuse_iodine")

    v_xa_28, v_xi_14 = make_b4(input_aceto,   "vessel_xa")
    v_xv_28, v_xv_14 = make_b4(input_vascular, "vessel_xv")
    v_xa_28p = conv_bn_relu(v_xi_28, C, name="v_xa_28p")
    v_xa_14p = conv_bn_relu(v_xi_14, C, name="v_xa_14p")
    v_xv_28p = conv_bn_relu(v_xv_28, C, name="v_xv_28p")
    v_xv_14p = conv_bn_relu(v_xv_14, C, name="v_xv_14p")
    feat_vessel = fuse(v_xa_28p, v_xa_14p, v_xv_28p, v_xv_14p,
                       name="fuse_vessel")


    m_xa_28, m_xa_14 = make_b0(input_aceto,  "margin_xa")
    m_xi_28, m_xi_14 = make_b0(input_iodine, "margin_xi")
    m_xa_28p = conv_bn_relu(m_xa_28, C, name="m_xa_28p")
    m_xa_14p = conv_bn_relu(m_xa_14, C, name="m_xa_14p")
    m_xi_28p = conv_bn_relu(m_xi_28, C, name="m_xi_28p")
    m_xi_14p = conv_bn_relu(m_xi_14, C, name="m_xi_14p")
    feat_margin = fuse(m_xa_28p, m_xa_14p, m_xi_28p, m_xi_14p,
                       name="fuse_margin")


    l_xa_28, l_xa_14 = make_b0(input_aceto,  "lesion_xa")
    l_xi_28, l_xi_14 = make_b0(input_iodine, "lesion_xi")
    l_xa_28p = conv_bn_relu(l_xa_28, C, name="l_xa_28p")
    l_xa_14p = conv_bn_relu(l_xa_14, C, name="l_xa_14p")
    l_xi_28p = conv_bn_relu(l_xi_28, C, name="l_xi_28p")
    l_xi_14p = conv_bn_relu(l_xi_14, C, name="l_xi_14p")
    feat_lesion = fuse(l_xa_28p, l_xa_14p, l_xi_28p, l_xi_14p,
                       name="fuse_lesion")

  
    return keras.Model(
        inputs=[input_aceto, input_iodine, input_vascular],
        outputs={
            "out_aceto":  head(feat_aceto,  "aceto"),
            "out_iodine": head(feat_iodine, "iodine"),
            "out_vessel": head(feat_vessel, "vessel"),
            "out_margin": head(feat_margin, "margin"),
            "out_lesion": head(feat_lesion, "lesion"),
        },
        name="SWEDE_SeparateExtraction"
    )

**Custom Loss Function**

In [ ]:
HEAD_NAMES  = ["out_aceto", "out_iodine", "out_vessel",
               "out_margin", "out_lesion"]
HEAD_LABELS = ["Aceto uptake", "Iodine uptake", "Vessel pattern",
               "Margin", "Lesion size"]
CLASS_NAMES = ["0", "1", "2"]
SCORE_VALUES = tf.constant([0.0, 1.0, 2.0])


def make_weighted_focal_loss(gamma=2.0):
    def loss_fn(y_true_ohe, y_pred, class_weight_tensor=None):
        y_pred_clipped = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce             = -y_true_ohe * tf.math.log(y_pred_clipped)
        p_t            = tf.reduce_sum(y_true_ohe * y_pred_clipped,
                                       axis=-1, keepdims=True)
        focal_weight   = (1.0 - p_t) ** gamma
        per_sample     = tf.reduce_sum(focal_weight * ce, axis=-1)
        if class_weight_tensor is not None:
            c_int          = tf.argmax(y_true_ohe, axis=-1)
            sample_weights = tf.gather(class_weight_tensor, c_int)
            per_sample     = per_sample * sample_weights
        return tf.reduce_mean(per_sample)
    return loss_fn


def make_total_score_loss():
   
    def loss_fn(labels, y_pred, y_total, delta=1.0):

        predicted   = []
        true_scores = []
        for h in HEAD_NAMES:
            pred_exp = tf.reduce_sum(y_pred[h] * SCORE_VALUES, axis=-1)
            true_exp = tf.reduce_sum(labels[h] * SCORE_VALUES, axis=-1)
            predicted.append(pred_exp)
            true_scores.append(true_exp)

        predicted_stacked   = tf.stack(predicted,   axis=1)  
        true_scores_stacked = tf.stack(true_scores, axis=1) 

        pred_total = tf.reduce_sum(predicted_stacked, axis=1)
        true_total = tf.cast(y_total, tf.float32)             

        total_error = pred_total - true_total
        abs_error   = tf.abs(total_error)
        huber       = tf.where(
            abs_error <= delta,
            0.5 * tf.square(total_error),
            delta * (abs_error - 0.5 * delta)
        )
        total_huber_loss = tf.reduce_mean(huber)

        head_errors = tf.abs(
            predicted_stacked - true_scores_stacked
        )                                                       
        error_sum   = tf.reduce_sum(
            head_errors, axis=1, keepdims=True
        ) + 1e-8
        weights     = head_errors / error_sum                  

        total_error_exp   = tf.expand_dims(total_error, 1)    
        head_corrections  = weights * total_error_exp         

        corrected_targets = tf.clip_by_value(
            true_scores_stacked - 0.1 * head_corrections,
            0.0, 2.0
        )

        head_correction_loss = tf.reduce_mean(
            tf.square(predicted_stacked - corrected_targets)
        )

        return total_huber_loss, head_correction_loss

    return loss_fn


class SaveBaseModelCheckpoint(keras.callbacks.Callback):
    def __init__(self, ckpt_path, monitor="val_loss"):
        super().__init__()
        self.ckpt_path = ckpt_path
        self.monitor   = monitor
        self.best      = np.inf

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is not None and current < self.best:
            self.best = current
            self.model.base_model.save_weights(self.ckpt_path)
            print(f"\n  Saved (val_loss={current:.4f}) → {self.ckpt_path}")


class HeadConvergenceCallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        self.model.update_head_convergence(logs or {})


class SWEDEModel(keras.Model):

    def __init__(self, base_model, gamma=2.0,
                 lambda_total=0.2,                             
                 class_weight_tensors=None,
                 convergence_patience=5,
                 convergence_delta=1e-3):
        super().__init__()
        self.base_model           = base_model
        self.gamma                = gamma
        self.lambda_total         = lambda_total
        self.loss_fn              = make_weighted_focal_loss(gamma=gamma)
        self.total_score_loss_fn  = make_total_score_loss()  
        self.class_weight_tensors = class_weight_tensors or {}
        self.convergence_patience = convergence_patience
        self.convergence_delta    = convergence_delta

        self._best_val_loss  = {h: np.inf for h in HEAD_NAMES}
        self._stagnant_count = {h: 0      for h in HEAD_NAMES}
        self._head_active    = {h: True   for h in HEAD_NAMES}

        self.loss_tracker       = keras.metrics.Mean(name="loss")
        self.total_score_tracker = keras.metrics.Mean(name="total_score_loss")  
        for h in HEAD_NAMES:
            setattr(self, f"tracker_{h}",
                    keras.metrics.Mean(
                        name=f"loss_{h.replace('out_', '')}"))

    def call(self, inputs, training=False):
        return self.base_model(inputs, training=training)

    def _head_loss(self, h, labels, y_pred):
        return self.loss_fn(
            labels[h],
            y_pred[h],
            self.class_weight_tensors.get(h, None)
        )

    def train_step(self, data):
        inputs, labels = data
        y_total = labels["y_total"]

        with tf.GradientTape() as tape:
            y_pred = self(inputs, training=True)

            head_losses = {
                h: self._head_loss(h, labels, y_pred)
                for h in HEAD_NAMES
                if self._head_active[h]
            }
            focal_loss = (tf.add_n(list(head_losses.values()))
                          if head_losses else tf.constant(0.0))

            
            huber_loss, correction_loss = self.total_score_loss_fn(
                labels, y_pred, y_total
            )
            total_score_loss = huber_loss + correction_loss

            total_loss = focal_loss + self.lambda_total * total_score_loss

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(
            (g, v) for g, v in zip(grads, self.trainable_variables)
            if g is not None
        )

        self.loss_tracker.update_state(total_loss)
        self.total_score_tracker.update_state(total_score_loss)
        for h in HEAD_NAMES:
            val = head_losses.get(h, tf.constant(0.0))
            getattr(self, f"tracker_{h}").update_state(val)

        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        inputs, labels = data
        y_total = labels["y_total"]

        y_pred = self(inputs, training=False)

        head_losses = {
            h: self._head_loss(h, labels, y_pred)
            for h in HEAD_NAMES
        }
        focal_loss = tf.add_n(list(head_losses.values()))

        huber_loss, correction_loss = self.total_score_loss_fn(
            labels, y_pred, y_total
        )
        total_score_loss = huber_loss + correction_loss
        total_loss       = focal_loss + self.lambda_total * total_score_loss

        self.loss_tracker.update_state(total_loss)
        self.total_score_tracker.update_state(total_score_loss)
        for h in HEAD_NAMES:
            getattr(self, f"tracker_{h}").update_state(head_losses[h])

        return {m.name: m.result() for m in self.metrics}

    def update_head_convergence(self, val_logs):
        for h in HEAD_NAMES:
            key      = f"val_loss_{h.replace('out_', '')}"
            val_loss = val_logs.get(key, None)
            if val_loss is None:
                continue
            if val_loss < self._best_val_loss[h] - self.convergence_delta:
                self._best_val_loss[h]  = val_loss
                self._stagnant_count[h] = 0
            else:
                self._stagnant_count[h] += 1
            if (self._stagnant_count[h] >= self.convergence_patience
                    and self._head_active[h]):
                self._head_active[h] = False
                print(f"\n  HEAD '{h}' converged → gradients FROZEN")

    @property
    def metrics(self):
        return ([self.loss_tracker, self.total_score_tracker] +
                [getattr(self, f"tracker_{h}") for h in HEAD_NAMES])

**Training**

In [ ]:
def compute_ens_class_weights(y_int, num_classes=3, beta=0.9999):
    counts = np.bincount(y_int, minlength=num_classes).astype(np.float32)

    effective_n = (1.0 - beta ** counts) / (1.0 - beta)
  
    weights = 1.0 / effective_n

    weights = weights / weights.sum() * num_classes

    print(f"  counts:     {counts.astype(int)}")
    print(f"  effective_n:{np.round(effective_n, 1)}")
    print(f"  weights:    {np.round(weights, 3)}")

    return tf.constant(weights, dtype=tf.float32)

In [ ]:
os.makedirs("/kaggle/working/models", exist_ok=True)

for fold_info in all_folds_data:
    fold = fold_info['fold']
    print(f"\n{'='*50}\n  FOLD {fold}\n{'='*50}")

    (xa_tr, xi_tr, xv_tr,
     ya_tr, yi_tr, yv_tr, ym_tr, yl_tr,
     ya_int_tr, yi_int_tr, yv_int_tr, ym_int_tr, yl_int_tr,
     y_total_tr) = fold_info['train']

    (xa_val, xi_val, xv_val,
     ya_val, yi_val, yv_val, ym_val, yl_val,
     y_total_val) = fold_info['val']

    train_ds = make_dataset(
        xa_tr, xi_tr, xv_tr,
        ya_tr, yi_tr, yv_tr, ym_tr, yl_tr,
        y_total_tr, batch_size=8, training=True
    )
    val_ds = make_dataset(
        xa_val, xi_val, xv_val,
        ya_val, yi_val, yv_val, ym_val, yl_val,
        y_total_val, batch_size=8, training=False
    )

    class_weight_tensors = {
        "out_aceto":  compute_ens_class_weights(ya_int_tr, beta=0.999),
        "out_iodine": compute_ens_class_weights(yi_int_tr, beta=0.99),
        "out_vessel": compute_ens_class_weights(yv_int_tr, beta=0.9999),
        "out_margin": compute_ens_class_weights(ym_int_tr, beta=0.999),
        "out_lesion": compute_ens_class_weights(yl_int_tr, beta=0.999),
    }

    base_model = build_swede_model(dropout_rate=0.5)
    model = SWEDEModel(
        base_model,
        gamma=2.0,
        lambda_total=0.2,                              
        class_weight_tensors=class_weight_tensors,
        convergence_patience=5,
        convergence_delta=1e-3,
    )
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=3.5e-5,
            weight_decay=1e-4
        )
    )

    ckpt_path = f"/kaggle/working/models/best_model_fold_{fold}.weights.h5"
    callbacks = [
        SaveBaseModelCheckpoint(ckpt_path, monitor="val_loss"),
        HeadConvergenceCallback(),
        keras.callbacks.EarlyStopping(
            monitor="val_loss", mode="min",
            patience=15, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", mode="min",
            factor=0.5, patience=5, min_lr=1e-7, verbose=1
        ),
    ]

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=50,
        callbacks=callbacks,
        verbose=1,
    )
    print(f"Fold {fold} done → {ckpt_path}")

# Result

In [ ]:
def run_ensemble_inference(xa_test, xi_test, xv_test,
                           ya_test, yi_test, yv_test,
                           ym_test, yl_test,
                           y_total_test,              
                           n_folds=3):
    test_ds = make_dataset(
        xa_test, xi_test, xv_test,
        ya_test, yi_test, yv_test, ym_test, yl_test,
        y_total_test,                                 
        batch_size=8, training=False
    )
    fold_probs = {h: [] for h in HEAD_NAMES}

    for fold_i in range(n_folds):
        model_path = f"/kaggle/working/models/best_model_fold_{fold_i}.weights.h5"
        print(f"Loading fold {fold_i}...")
        m = build_swede_model()
        m.load_weights(model_path)
        preds = m.predict(test_ds, verbose=0)
        for h in HEAD_NAMES:
            fold_probs[h].append(preds[h])

    ensemble_probs = {
        h: np.mean(fold_probs[h], axis=0) for h in HEAD_NAMES
    }
    y_true_dict = {
        "out_aceto":  np.argmax(ya_test, axis=1),
        "out_iodine": np.argmax(yi_test, axis=1),
        "out_vessel": np.argmax(yv_test, axis=1),
        "out_margin": np.argmax(ym_test, axis=1),
        "out_lesion": np.argmax(yl_test, axis=1),
    }
    return ensemble_probs, y_true_dict

In [ ]:
def evaluate_head(head_name, display_name, y_true_int, y_prob):
    y_pred    = np.argmax(y_prob, axis=1)
    accuracy  = accuracy_score(y_true_int, y_pred)
    precision = precision_score(y_true_int, y_pred,
                                average='macro', zero_division=0)
    recall    = recall_score(y_true_int, y_pred,
                             average='macro', zero_division=0)
    f1        = f1_score(y_true_int, y_pred,
                         average='macro', zero_division=0)
    mae       = mean_absolute_error(y_true_int, y_pred)
    try:
        auc = roc_auc_score(y_true_int, y_prob,
                            multi_class='ovr', average='macro')
    except ValueError:
        auc = float('nan')

    cm = confusion_matrix(y_true_int, y_pred)

    print(f"\n{'='*45}")
    print(f"  {display_name.upper()}")
    print(f"{'='*45}")
    print(f"  Accuracy  : {accuracy:.4f}")
    print(f"  Precision : {precision:.4f}  (macro)")
    print(f"  Recall    : {recall:.4f}  (macro)")
    print(f"  F1 Score  : {f1:.4f}  (macro)")
    print(f"  AUC (OvR) : {auc:.4f}")
    print(f"  MAE       : {mae:.4f}")
    print(classification_report(y_true_int, y_pred,
                                target_names=CLASS_NAMES,
                                zero_division=0))
    return {
        "head": head_name, "display": display_name,
        "accuracy": accuracy, "precision": precision,
        "recall": recall, "f1": f1, "auc": auc,
        "mae": mae, "cm": cm,
        "y_true": y_true_int, "y_pred": y_pred, "y_prob": y_prob,
    }


def plot_all_confusion_matrices(results):
    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    fig.suptitle("Confusion matrices — ensemble predictions",
                 fontsize=13, y=1.02)
    for ax, r in zip(axes, results):
        cm_norm  = r["cm"].astype(float)
        row_sums = cm_norm.sum(axis=1, keepdims=True)
        cm_norm  = np.divide(cm_norm, row_sums, where=row_sums != 0)
        sns.heatmap(
            cm_norm, annot=r["cm"], fmt="d", cmap="Blues",
            vmin=0, vmax=1,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, cbar=False, linewidths=0.5,
        )
        ax.set_title(
            f"{r['display']}\nF1={r['f1']:.3f}  AUC={r['auc']:.3f}",
            fontsize=10)
        ax.set_xlabel("Predicted", fontsize=9)
        ax.set_ylabel("True",      fontsize=9)
        ax.tick_params(axis='both', labelsize=8)
    plt.tight_layout()
    plt.savefig("/kaggle/working/confusion_matrices.png",
                dpi=150, bbox_inches="tight")
    plt.show()


def print_summary_table(results):
    print(f"\n{'='*70}")
    print(f"  SUMMARY — all 5 SWEDE scores")
    print(f"{'='*70}")
    print(f"  {'Score':<18} {'Acc':>6} {'Prec':>6} "
          f"{'Rec':>6} {'F1':>6} {'AUC':>6} {'MAE':>6}")
    print(f"  {'-'*60}")
    for r in results:
        print(f"  {r['display']:<18} "
              f"{r['accuracy']:>6.3f} {r['precision']:>6.3f} "
              f"{r['recall']:>6.3f} {r['f1']:>6.3f} "
              f"{r['auc']:>6.3f} {r['mae']:>6.3f}")
    print(f"{'='*70}")



(xa_test, xi_test, xv_test,
 ya_test, yi_test, yv_test,
 ym_test, yl_test,
 y_total_test) = all_folds_data[0]["test"]

ensemble_probs, y_true_dict = run_ensemble_inference(
    xa_test, xi_test, xv_test,
    ya_test, yi_test, yv_test, ym_test, yl_test,
    y_total_test,                                      
    n_folds=3
)

results = []
for h_name, h_label in zip(HEAD_NAMES, HEAD_LABELS):
    r = evaluate_head(h_name, h_label, y_true_dict[h_name],
                      ensemble_probs[h_name])
    results.append(r)

plot_all_confusion_matrices(results)
print_summary_table(results)